# 06 - Simple ML Forecast Baselines

In this notebook, we begin the machine learning side of the project with simple, interpretable forecast baselines.

The goal is not to build the most advanced model first. The goal is to create a careful bridge from decline curve analysis into supervised learning.

Early models in this notebook:

- Naive last-observed oil baseline
- Trailing average oil baseline
- Simple feature table using only information available at forecast time
- Later: linear regression and possibly random forest, depending on installed libraries

Important project framing carried forward from the DCA notebooks:

- Target commodity: oil production
- Gas forecasting: excluded from release 1
- DCA fit window: months 12-24
- DCA forecast-check window: months 25-33
- Fair ML comparison rule: do not use future production months as input features

## Target Decision: Same-Month Prediction vs. True Forecasting

Before writing model code, we need to decide what the ML target means.

A same-month prediction setup would ask:

> Given information from month `m`, predict oil production in month `m`.

That is usually not a forecasting problem. If the feature table accidentally includes same-month oil, same-month normalized production, or variables only known after the month closes, the model can look better than it really is.

A true one-step-ahead setup asks:

> Given information available through month `m`, predict oil production in month `m + 1`.

That is the safer starting point for ML. For example, month 24 information can be used to forecast month 25.

There is one more fairness detail:

- A rolling one-step forecast for months 25-33 may use observed month 25 to forecast month 26, observed month 26 to forecast month 27, and so on.
- The DCA benchmark is stricter: it forecasts months 25-33 from a month-24 origin.

So this notebook starts by building the one-step-ahead table. Later, when we compare directly against DCA, we will be explicit about whether the ML baseline is rolling one-step or fixed-origin from month 24.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

Line-by-line explanation:

- `Path` helps us build file paths that work cleanly across folders.
- `numpy` gives us numerical helpers.
- We will import plotting tools later, when we actually build the first chart.
- `pandas` is the main tool for reading the monthly production table and building ML-ready rows.

In [ ]:
current_folder = Path.cwd()

candidate_roots = [current_folder, *current_folder.parents]

PROJECT_ROOT = next(
    root
    for root in candidate_roots
    if (root / "data" / "processed").exists()
)

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "martin_selected_30_monthly_production_normalized.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "ml_outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE

Line-by-line explanation:

- `current_folder = Path.cwd()` records the folder where the notebook is currently running.
- `candidate_roots` checks that folder and each parent folder.
- `PROJECT_ROOT = next(...)` finds the first folder that contains `data/processed`.
- `DATA_FILE` points to the cleaned monthly production file used in the DCA workflow.
- `OUTPUT_DIR` creates a separate report folder for ML outputs.
- `mkdir(..., exist_ok=True)` makes the output folder if it does not already exist.
- The final line displays the data path so we can verify that the notebook found the right file.

In [ ]:
df = pd.read_csv(
    DATA_FILE,
    dtype={
        "api8": str,
        "lease_no": str,
        "district": str,
    },
)

df.head()

Line-by-line explanation:

- `pd.read_csv(...)` loads the processed monthly production table.
- The `dtype` settings keep ID-style fields as text, which prevents leading-zero or formatting surprises.
- `df.head()` shows the first few rows so we can inspect the shape of the data before transforming it.

In [ ]:
numeric_columns = [
    "month_on_production",
    "oil_bbl",
    "casinghead_gas_mcf",
    "boe",
    "interval_length_proxy_ft",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.sort_values(["api8", "month_on_production"]).reset_index(drop=True)

df.head()

Line-by-line explanation:

- `numeric_columns` lists the fields that should behave like numbers.
- The `for` loop converts each listed column to numeric values.
- `errors="coerce"` turns unexpected non-numeric values into missing values instead of crashing.
- Sorting by `api8` and `month_on_production` puts every well in time order.
- `reset_index(drop=True)` gives the sorted table a clean row index.

In [ ]:
cohort_summary = pd.Series(
    {
        "row_count": len(df),
        "well_count": df["api8"].nunique(),
        "first_month_on_production": df["month_on_production"].min(),
        "last_month_on_production": df["month_on_production"].max(),
        "last_dca_forecast_check_month": 33,
    }
)

cohort_summary

Line-by-line explanation:

- `pd.Series({...})` creates a compact summary we can read like a checklist.
- `row_count` tells us how many monthly well records are available.
- `well_count` confirms the size of the selected Martin County cohort.
- The first and last month fields show the available production history.
- `last_dca_forecast_check_month` reminds us that the first ML comparison should stay aligned with the DCA forecast-check window.

In [ ]:
modeling_df = df[df["month_on_production"].between(1, 33)].copy()

modeling_df.shape

Line-by-line explanation:

- `between(1, 33)` keeps the same early-life window used for the DCA benchmark comparison.
- `.copy()` gives us a separate table for ML feature work.
- `.shape` reports the number of rows and columns after the filter.

In [ ]:
well_groups = modeling_df.groupby("api8", group_keys=False)

modeling_df["target_month_on_production"] = well_groups["month_on_production"].shift(-1)
modeling_df["target_next_oil_bbl"] = well_groups["oil_bbl"].shift(-1)

modeling_df["last_observed_oil_bbl"] = modeling_df["oil_bbl"]
modeling_df["trailing_3mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)
modeling_df["trailing_6mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

one_step_rows = modeling_df.dropna(
    subset=[
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
    ]
).copy()

one_step_rows[
    [
        "api8",
        "month_on_production",
        "oil_bbl",
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
        "interval_length_proxy_ft",
    ]
].head(10)

Line-by-line explanation:

- `groupby("api8")` makes sure lags and targets are created within each well, never across wells.
- `shift(-1)` moves next month values back onto the current row.
- `target_next_oil_bbl` is the value we want to forecast.
- `last_observed_oil_bbl` is the simplest possible forecast input: the oil volume from the current month.
- The rolling averages use only current and earlier observed months inside each well.
- `dropna(...)` removes rows where the next-month target is missing.
- The final display shows the current month, next-month target, and first simple baseline features side by side.

In [ ]:
train_rows = one_step_rows[one_step_rows["target_month_on_production"].between(13, 24)].copy()
forecast_check_rows = one_step_rows[one_step_rows["target_month_on_production"].between(25, 33)].copy()

train_rows.shape, forecast_check_rows.shape

Line-by-line explanation:

- `train_rows` keeps examples whose target months are 13-24.
- `forecast_check_rows` keeps examples whose target months are 25-33.
- This mirrors the DCA split at a high level, but it is still a rolling one-step table.
- For a strict fixed-origin comparison against DCA, we will later forecast months 25-33 using only information available through month 24.

## Next Block To Build

The next manual step should be the naive last-observed oil baseline.

That baseline will answer:

> If the forecast for next month is simply this month's oil production, how wrong are we over the forecast-check months?

Before treating it as a DCA competitor, we will label it clearly as a rolling one-step baseline unless we freeze the forecast origin at month 24.